# Sumarização por Ramo — Bases Anonimizadas SUSEP

Gera **3 tabelas** (uma por ramo), cada uma com merge R_ ↔ S_ seguido de agrupamento geográfico:

| Ramo | Merge R↔S por | Agrupamento | Colunas de prêmio | Colunas de sinistro |
|------|---------------|-------------|--------------------|---------|
| **auto** | `cod_apo, cod_endosso, item, regiao` | `cep_per` | `premio_total_auto` | `indeniz` |
| **comp** | `cod_apo, cod_endosso, tipo, classe, item, cobertura, uf` | `uf` | `premio` | `indeniz` |
| **rural** | `cod_apo, cod_endosso, cod_item, cobertura, cob_fundo, cod_mod, cultura, munic, uf, id_bem` | `munic` (+`uf`) | `premio` | `indeniz`, `desp_sin` |

### Abordagem
1. Para cada ramo, filtrar os datasets harmonizados de riscos e sinistros
2. Agregar prêmios e sinistros separadamente pela chave geográfica
3. Fazer merge dos agregados (outer join: todos os locais com prêmio e/ou sinistro)
4. Calcular sinistralidade

### Engine
- **Polars lazy** (`scan_parquet` → `filter` → `group_by` → `agg` → `collect`) — nunca materializa os ~235M linhas
- **Pandas** apenas para exibição das tabelas finais já agregadas (poucos milhares de linhas)

## 1 — Setup e Configuração

In [ ]:
import polars as pl
from pathlib import Path

UNIFIED_ROOT = Path("data/unified")
RISCOS_DIR = UNIFIED_ROOT / "parquet" / "harmonizado_riscos"
SINISTROS_DIR = UNIFIED_ROOT / "parquet" / "harmonizado_sinistros"
OUT_DIR = UNIFIED_ROOT / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Chaves de integração R↔S (conforme PLANO seção E.3) ────────────────
CHAVES_MERGE = {
    "auto":  ["cod_apo", "cod_endosso", "item", "regiao"],
    "comp":  ["cod_apo", "cod_endosso", "tipo", "classe", "item", "cobertura", "uf"],
    "rural": ["cod_apo", "cod_endosso", "cod_item", "cobertura", "cob_fundo",
              "cod_mod", "cultura", "munic", "uf", "id_bem"],
}

# ── Coluna geográfica de agrupamento por ramo ──────────────────────────
GEO_COL = {
    "auto":  "cep_per",     # CEP de pernoite do veículo (riscos)
    "comp":  "uf",          # UF do risco
    "rural": "munic",       # Código do município
}

# Nos sinistros a coluna pode ter nome diferente → renomear para GEO_COL
GEO_COL_SIN = {
    "auto":  "cep",         # S_AUTO tem 'cep' em vez de 'cep_per'
    "comp":  "uf",
    "rural": "munic",
}

# ── Coluna(s) extra de contexto geográfico na saída ───────────────────
GEO_EXTRA = {
    "auto":  [],
    "comp":  [],
    "rural": ["uf"],
}

# ── Coluna de prêmio por ramo ─────────────────────────────────────────
PREMIO_COL = {
    "auto":  "premio_total_auto",
    "comp":  "premio",
    "rural": "premio",
}

# ── Colunas de sinistro por ramo ──────────────────────────────────────
SINISTRO_COLS = {
    "auto":  ["indeniz"],
    "comp":  ["indeniz"],
    "rural": ["indeniz", "desp_sin"],
}

print(f"✓ Setup — Polars {pl.__version__}")
print(f"  Riscos:    {RISCOS_DIR}")
print(f"  Sinistros: {SINISTROS_DIR}")
for ramo in ["auto", "comp", "rural"]:
    print(f"  {ramo:6}: merge por {CHAVES_MERGE[ramo]} → agrupar por {GEO_COL[ramo]}")

✓ Setup — Polars 1.39.0
  Riscos:    data/unified/parquet/harmonizado_riscos
  Sinistros: data/unified/parquet/harmonizado_sinistros
  auto  : merge por ['cod_apo', 'cod_endosso', 'item', 'regiao'] → agrupar por cep_per
  comp  : merge por ['cod_apo', 'cod_endosso', 'tipo', 'classe', 'item', 'cobertura', 'uf'] → agrupar por uf
  rural : merge por ['cod_apo', 'cod_endosso', 'cod_item', 'cobertura', 'cob_fundo', 'cod_mod', 'cultura', 'munic', 'uf', 'id_bem'] → agrupar por munic


## 2 — Funções de Agregação com Polars Lazy

Funções reutilizáveis que usam `pl.scan_parquet()` (lazy) para filtrar por ramo
e agregar pela chave geográfica sem materializar os dados completos em RAM.

In [ ]:
def _glob_ramo(base_dir: Path, ramo: str, prefixo: str) -> str:
    """Retorna glob para arquivos do ramo: ex. R_AUTO_*.parquet"""
    tag = ramo.upper()
    return str(base_dir / f"{prefixo}_{tag}_*.parquet")


def agregar_premios(ramo: str) -> pl.LazyFrame:
    """Lazy scan de harmonizado_riscos (só arquivos do ramo) → group_by geográfico → soma prêmio."""
    geo = GEO_COL[ramo]
    extras = GEO_EXTRA[ramo]
    premio_col = PREMIO_COL[ramo]
    group_cols = [geo] + extras

    lf = (
        pl.scan_parquet(_glob_ramo(RISCOS_DIR, ramo, "R"))
        .filter(pl.col(geo).is_not_null())
        .group_by(group_cols)
        .agg(
            pl.col(premio_col).sum().alias("premio"),
            pl.col(premio_col).count().alias("n_apolices"),
        )
    )
    return lf


def agregar_sinistros(ramo: str) -> pl.LazyFrame:
    """Lazy scan de harmonizado_sinistros → join com riscos (via CHAVES_MERGE)
    para herdar a variável geográfica do risco → group_by geográfico → soma sinistro.

    Mesma lógica para todos os ramos:
      auto:  join herda cep_per (não existe em S_AUTO)
      comp:  join por chaves que já incluem uf
      rural: join por chaves que já incluem munic + uf
    """
    geo = GEO_COL[ramo]
    extras = GEO_EXTRA[ramo]
    sin_cols = SINISTRO_COLS[ramo]
    group_cols = [geo] + extras
    merge_keys = CHAVES_MERGE[ramo]

    agg_exprs = [
        pl.col(c).sum().alias(c) for c in sin_cols
    ] + [
        pl.len().alias("n_sinistros"),
    ]

    lf_sin = pl.scan_parquet(_glob_ramo(SINISTROS_DIR, ramo, "S"))

    # Colunas geográficas que precisam vir dos riscos (não estão nas chaves de merge)
    geo_extras_risco = [c for c in [geo] + extras if c not in merge_keys]
    lf_risco = (
        pl.scan_parquet(_glob_ramo(RISCOS_DIR, ramo, "R"))
        .select(merge_keys + geo_extras_risco)
    )

    lf = (
        lf_sin
        .join(lf_risco, on=merge_keys, how="left")
        .filter(pl.col(geo).is_not_null())
        .group_by(group_cols)
        .agg(agg_exprs)
    )
    return lf


def merge_e_agrupar(ramo: str) -> pl.DataFrame:
    """Agrega prêmios e sinistros via Polars lazy, faz join pelo campo geográfico."""
    geo = GEO_COL[ramo]
    extras = GEO_EXTRA[ramo]
    join_keys = [geo] + extras

    print(f"\n{'='*60}")
    print(f"RAMO: {ramo.upper()} — agrupar por '{geo}'")
    print(f"  Chaves merge R↔S: {CHAVES_MERGE[ramo]}")
    print(f"{'='*60}")

    print(f"  Agregando prêmios...")
    lf_premio = agregar_premios(ramo)
    df_premio = lf_premio.collect()
    print(f"  Riscos {ramo}: → {len(df_premio):,} chaves geográficas")

    print(f"  Agregando sinistros...")
    lf_sinistro = agregar_sinistros(ramo)
    df_sinistro = lf_sinistro.collect()
    print(f"  Sinistros {ramo}: → {len(df_sinistro):,} chaves geográficas")

    # Join outer prêmio ↔ sinistro
    if len(df_premio) == 0 and len(df_sinistro) == 0:
        print(f"  ⚠ Nenhum dado para ramo {ramo}")
        return pl.DataFrame()

    if len(df_sinistro) == 0:
        df = df_premio.with_columns(
            *[pl.lit(0.0).alias(c) for c in SINISTRO_COLS[ramo]],
            pl.lit(0).cast(pl.UInt32).alias("n_sinistros"),
        )
    elif len(df_premio) == 0:
        df = df_sinistro.with_columns(
            pl.lit(0.0).alias("premio"),
            pl.lit(0).cast(pl.UInt32).alias("n_apolices"),
        )
    else:
        df = df_premio.join(df_sinistro, on=join_keys, how="full", coalesce=True)
        # Preencher nulls do join
        df = df.with_columns(
            pl.col("premio").fill_null(0.0),
            pl.col("n_apolices").fill_null(0),
            *[pl.col(c).fill_null(0.0) for c in SINISTRO_COLS[ramo]],
            pl.col("n_sinistros").fill_null(0),
        )

    # Sinistro total e sinistralidade
    sin_total_expr = sum(pl.col(c) for c in SINISTRO_COLS[ramo])
    df = df.with_columns(
        sin_total_expr.alias("sinistro_total"),
    ).with_columns(
        pl.when(pl.col("premio") > 0)
        .then((pl.col("sinistro_total") / pl.col("premio")).round(4))
        .otherwise(None)
        .alias("sinistralidade"),
    ).sort(geo)

    total_p = df["premio"].sum()
    total_s = df["sinistro_total"].sum()

    print(f"\n  ✓ Tabela final: {len(df):,} linhas | colunas: {df.columns}")
    print(f"    Prêmio total:   R$ {total_p:>18,.2f}")
    print(f"    Sinistro total: R$ {total_s:>18,.2f}")
    if total_p > 0:
        print(f"    Sinistralidade: {total_s / total_p:.2%}")

    return df

print("✓ Funções de agregação Polars definidas")

✓ Funções de agregação Polars definidas


## 3 — Tabela AUTO: Merge R↔S → Agrupamento por CEP de Pernoite

- **Merge** por: `cod_apo`, `cod_endosso`, `item`, `regiao`
- **Agrupamento** por: `cep_per` (CEP de pernoite do veículo)
- **Prêmio**: `premio_total_auto` (soma das coberturas pre_*)
- **Sinistro**: `indeniz`

In [10]:
df_auto = merge_e_agrupar("auto")
df_auto.head(20).to_pandas()


RAMO: AUTO — agrupar por 'cep_per'
  Chaves merge R↔S: ['cod_apo', 'cod_endosso', 'item', 'regiao']
  Agregando prêmios...
  Riscos auto: → 750,338 chaves geográficas
  Agregando sinistros...


ColumnNotFoundError: unable to find column "cep_per"; valid columns: ["cod_apo", "cod_endosso", "data_comp", "item", "modalidade", "tipo_prod", "cobertura", "cod_modelo", "ano_modelo", "cod_tarif", "regiao", "cod_cont", "evento", "indeniz", "val_salvad", "d_salvado", "val_ress", "d_ress", "d_avi", "d_liq", "d_ocorr", "causa", "sexo", "d_nasc", "cep", "ramo", "grupo_arquivo", "ano", "semestre", "arquivo_origem", "tipo", "classe", "uf", "val_franq", "d_aviso", "cod_item", "cob_fundo", "cod_mod", "id_bem", "cultura", "munic", "desp_sin", "ev_ger", "d_ocorr_in", "d_ocorr_fi", "fonte_susep_subpagina", "periodo_analitico"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'group_by' <---
Parquet SCAN [data/unified/parquet/harmonizado_sinistros/S_AUTO_auto_000.parquet, data/unified/parquet/harmonizado_sinistros/S_AUTO_auto_001.parquet]
PROJECT */47 COLUMNS
ESTIMATED ROWS: 6279950

This error occurred with the following context stack:
	[1] 'group_by'


## 4 — Tabela COMP: Merge R↔S → Agrupamento por UF

- **Merge** por: `cod_apo`, `cod_endosso`, `tipo`, `classe`, `item`, `cobertura`, `uf`
- **Agrupamento** por: `uf`
- **Prêmio**: `premio`
- **Sinistro**: `indeniz`

In [ ]:
df_comp = merge_e_agrupar("comp")
df_comp.head(30).to_pandas()

## 5 — Tabela RURAL: Merge R↔S → Agrupamento por Município

- **Merge** por: `cod_apo`, `cod_endosso`, `cod_item`, `cobertura`, `cob_fundo`, `cod_mod`, `cultura`, `munic`, `uf`, `id_bem`
- **Agrupamento** por: `munic` (+`uf`)
- **Prêmio**: `premio`
- **Sinistro**: `indeniz` + `desp_sin`

In [ ]:
df_rural = merge_e_agrupar("rural")
df_rural.head(20).to_pandas()

## 6 — Exportação

In [ ]:
exportados = {}

for ramo, df in [("auto", df_auto), ("comp", df_comp), ("rural", df_rural)]:
    if len(df) == 0:
        print(f"  ⚠ {ramo}: sem dados, pulando exportação")
        continue
    out_file = OUT_DIR / f"sumarizacao_{ramo}_por_{GEO_COL[ramo]}.csv"
    df.write_csv(str(out_file))
    exportados[ramo] = out_file
    print(f"  ✓ {out_file.name:50} ({len(df):>6,} linhas)")

print(f"\n{'='*60}")
print("RESUMO GERAL")
print(f"{'='*60}")
for ramo, df in [("auto", df_auto), ("comp", df_comp), ("rural", df_rural)]:
    if len(df) == 0:
        continue
    geo = GEO_COL[ramo]
    total_p = df["premio"].sum()
    total_s = df["sinistro_total"].sum()
    print(f"\n  {ramo.upper()} (por {geo}):")
    print(f"    {len(df):>6,} chaves geográficas")
    print(f"    Prêmio:       R$ {total_p:>18,.2f}")
    print(f"    Sinistro:     R$ {total_s:>18,.2f}")
    if total_p > 0:
        print(f"    Sinistralidade: {total_s / total_p:.2%}")
    print(f"    Arquivo: {exportados.get(ramo, 'N/A')}")